
## import necessary python packages

In [20]:
import os
import json
from openai import OpenAI
import random
import base64
import requests
import time

api_key = 'YOUR API KEY'

## import annotations and qa_pairs

In [21]:
with open("toalt_test_qa_pairs.json", 'r') as file:
    test_qa_pairs = json.load(file)
with open("test_annotations.json", 'r') as file:
    test_annotations = json.load(file)

## 向GPT-4V发起请求

In [27]:
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

def get_gpt_4v_reply(image_url, final_question):
    base64_image = encode_image(image_url)
    payload = {
      "model": "gpt-4-vision-preview",
      "messages": [
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": final_question
            },
            {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/png;base64,{base64_image}",
                'detail':'high'
              },
            }
          ]
        }
      ],
      "max_tokens": 1000
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
    # gpt_4v_total_tokens = response.json()['usage']['total_tokens']
    gpt_4v_reply = response.json()['choices'][0]['message']
    return  gpt_4v_reply

## main函数

In [ ]:

start_length = 304
total_questions = 0
gap = False
annotation_id = 23
total_question_limit = 200
for _, each_annotation in enumerate(test_annotations[annotation_id:]):
    total_charts = []
    if gap == True:
        break
    # print(each_annotation)
    begin = time.time()
    image_index = each_annotation['image']
    # print(image_index)
    image_url = 'qaBench/test/charts/original/' + image_index
    image_type = each_annotation['type']
    index = each_annotation['id']
    for i, each_qa_pair in enumerate(test_qa_pairs[start_length:]):
        each_chart = {}
        each_chart['image_url'] = image_url
        each_chart['image_type'] = image_type
        task_category = each_qa_pair['type']
        each_chart['task_category'] = task_category
        print(each_qa_pair['QA_pairs'][0]['fill_the_blank'])
        if index == each_qa_pair['image_index']:
            start_length += 1
            total_questions += 4
            # print(total_questions)
            print(f"{index}, {task_category}")
            for each_question_format in each_qa_pair['QA_pairs']:
                each_key = list(each_question_format.keys())[0]

                final_question = each_question_format[each_key][0]
                annotation = each_question_format[each_key][1]


                gpt_4v_reply = get_gpt_4v_reply(image_url, final_question)

                print(f"第{index}张图片的{each_key}问题。")
                each_chart[each_key + ' question'] = final_question
                each_chart[each_key + ' annotation'] = annotation
                each_chart[each_key + ' GPT-4v'] = gpt_4v_reply
            each_chart['start_length'] = start_length
            each_chart['annotation_id'] = annotation_id
            each_chart['pair_index'] = each_qa_pair['pair_index']
            total_charts.append(each_chart)
        else:
            break
        if total_questions == total_question_limit:
            gap = True
            print(f"本次测试结束在第{annotation_id}/{len(test_annotations)}张图片, 第{start_length}/{len(test_qa_pairs)}个问题")
            break
    end = time.time()
    each_annotation_time = round(end - begin,2)
    annotation_id += 1
    print(f"第{annotation_id}张图片一共耗时{each_annotation_time}。")
